In [1]:
# Install required packages

!pip install -q pdf2docx python-docx pandas openpyxl openai tqdm

In [2]:
# Allow access to google drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Access and download online pdf file of cyber incidents

import requests

# === STEP 1: Download the PDF ===
pdf_url = 'https://csis-website-prod.s3.amazonaws.com/s3fs-public/2025-05/250502_Significant_Cyber_Events.pdf'  # Replace with the actual URL
pdf_path = 'latest.pdf'

response = requests.get(pdf_url)
with open(pdf_path, 'wb') as f:
    f.write(response.content)


In [4]:
# Convert pdf file to docx

# Step 1: Install dependencies
!pip install pdf2docx python-docx

# Step 2: Convert PDF to DOCX
from pdf2docx import Converter

pdf_path = 'latest.pdf'
docx_path = 'latest.docx'

converter = Converter(pdf_path)
converter.convert(docx_path, start=0, end=None)
converter.close()


In [6]:
# Clean docx file

from docx import Document
import re

def clean_and_merge_docx(input_path, output_path):
    # Load original DOCX
    doc = Document(input_path)
    raw_paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

    # Remove boilerplate
    boilerplate = "Center for Strategic and International Studies (CSIS) | Washington, D.C."
    cleaned_paragraphs = [p.replace(boilerplate, '').strip() for p in raw_paragraphs if p.replace(boilerplate, '').strip()]

    # Regex patterns
    month_season_variants = r'(January|February|March|Mach|April|May|June|July|August|September|October|November|December|Spring|Summer|Fall|Winter)'
    valid_heading_pattern = re.compile(
        rf'^({month_season_variants}(?:\s*[–-]\s*{month_season_variants})?\s*20\d{{2}})[\s:\-–.]*',
        flags=re.IGNORECASE
    )
    # Match lines that start with a 4-digit year (e.g., "2007.")
    year_only_pattern = re.compile(r'^\d{4}[\s\.\-–:]')

    # Merge paragraphs
    merged_paragraphs = []
    for para in cleaned_paragraphs:
        if valid_heading_pattern.match(para) or year_only_pattern.match(para):
            merged_paragraphs.append(para)
        elif merged_paragraphs:
            merged_paragraphs[-1] += ' ' + para
        else:
            continue  # Skip orphan text

    # Write to new DOCX
    new_doc = Document()
    for para in merged_paragraphs:
        new_doc.add_paragraph(para)

    new_doc.save(output_path)
    print(f"✅ Cleaned DOCX saved to: {output_path}")


# Call the function
clean_and_merge_docx('latest.docx', 'cleaned_latest.docx')

✅ Cleaned DOCX saved to: cleaned_latest.docx


In [7]:
from docx import Document
import pandas as pd
import os

def process_docx_to_excel(docx_path, master_path, new_data_path):
    # Step 1: Extract cleaned paragraphs from DOCX
    doc = Document(docx_path)
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    df_new = pd.DataFrame(paragraphs, columns=["Event Description"])

    if not os.path.exists(master_path):
        # First-time setup
        df_new.to_excel(master_path, index=False)
        df_new.to_excel(new_data_path, index=False)
        print(f"🆕 First run: Saved {len(df_new)} rows to both master and new_data.")
    else:
        # Load existing master data
        df_master = pd.read_excel(master_path)
        master_texts = set(df_master['Event Description'].dropna().str.strip())

        # Find new entries only
        df_unique_new = df_new[~df_new['Event Description'].str.strip().isin(master_texts)].copy()

        if df_unique_new.empty:
            print("✅ No new events found. Everything is already in the master.")
        else:
            # Save only new entries to new_data
            df_unique_new.to_excel(new_data_path, index=False)
            print(f"➕ Found and saved {len(df_unique_new)} new rows to: {new_data_path}")

            # Update and save master
            df_updated_master = pd.concat([df_master, df_unique_new], ignore_index=True)
            df_updated_master.drop_duplicates(subset=["Event Description"], inplace=True)
            df_updated_master.to_excel(master_path, index=False)
            print(f"📦 Master updated and saved with {len(df_updated_master)} total rows.")



In [8]:
docx_path = 'cleaned_latest.docx'
master_path = '/content/drive/MyDrive/Colab Notebooks/CyberIncidents/cyber_master_data.xlsx'
new_data_path = '/content/drive/MyDrive/Colab Notebooks/CyberIncidents/new_data.xlsx'

process_docx_to_excel(docx_path, master_path, new_data_path)

➕ Found and saved 2 new rows to: /content/drive/MyDrive/Colab Notebooks/CyberIncidents/new_data.xlsx
📦 Master updated and saved with 1159 total rows.


In [9]:
# Convert April2020 to April 2020 and Mach to March
def clean_text(text):
    if pd.isna(text):
        return text
    # Fix April2020 (case insensitive, allow optional space)
    text = re.sub(r'\bApril\s?2020\b', 'April 2020', text, flags=re.IGNORECASE)
    # Fix "Mach" as a standalone word (case insensitive)
    text = re.sub(r'(?<![a-zA-Z])Mach(?![a-zA-Z])', 'March', text, flags=re.IGNORECASE)
    return text.strip()

def clean_excel_file(file_path):
    df = pd.read_excel(file_path)
    if 'Event Description' in df.columns:
        df['Event Description'] = df['Event Description'].apply(clean_text)
        df.to_excel(file_path, index=False)
        print(f"✅ Cleaned and saved: {file_path}")
    else:
        print(f"⚠️ 'Event Description' column not found in {file_path}")

# Clean both files
clean_excel_file(master_path)
clean_excel_file(new_data_path)

✅ Cleaned and saved: /content/drive/MyDrive/Colab Notebooks/CyberIncidents/cyber_master_data.xlsx
✅ Cleaned and saved: /content/drive/MyDrive/Colab Notebooks/CyberIncidents/new_data.xlsx


We have two excel files downloaded to the folder; 'cyber_master_data' and 'new_data'.
'cyber_master_data.xlsx' is for comparison purposes only. When we access the online pdf file, the code compares extracted new entries against this file. If there are new entries, they are appended to 'cyber_master_data.xlsx' and also saved as 'new_data'.

In [10]:
# check new_data.xlsx contains only new data not processed before

new_data_path = '/content/drive/MyDrive/Colab Notebooks/CyberIncidents/new_data.xlsx'
existing_path = "/content/drive/MyDrive/Colab Notebooks/CyberIncidents/cyber-incidents-final.xlsx"
news_path = "/content/drive/MyDrive/Colab Notebooks/CyberIncidents/news.xlsx"
input_path = "/content/drive/MyDrive/Colab Notebooks/CyberIncidents/non-parsed.xlsx"

# Step 1: Load new_data.xlsx (must exist)
df_raw = pd.read_excel(new_data_path, header=0)
df_raw.dropna(inplace=True)
df_raw[df_raw.columns[0]] = df_raw[df_raw.columns[0]].astype(str).str.strip()

# Step 2: Initialize set of existing entries from other files
existing_entries = set()

# Step 3: Function to extract and clean entries from a given file
def load_existing_entries(file_path, col_index=0):
    if os.path.exists(file_path):
        df = pd.read_excel(file_path, header=0)
        if df.shape[1] > col_index:
            entries = df.iloc[:, col_index].dropna().astype(str).str.strip()
            return set(entries)
        else:
            print(f"⚠️ File {file_path} does not contain expected column index {col_index}. Skipped.")
            return set()
    else:
        print(f"⚠️ File not found: {file_path}")
        return set()

# Step 4: Accumulate all existing entries from other sources
existing_entries |= load_existing_entries(existing_path, col_index=0)
existing_entries |= load_existing_entries(news_path)
existing_entries |= load_existing_entries(input_path)

# Step 5: Filter df_raw against existing entries
original_count = len(df_raw)
df_raw = df_raw[~df_raw[df_raw.columns[0]].isin(existing_entries)].reset_index(drop=True)
new_count = len(df_raw)

# Step 6: Save filtered result back to new_data.xlsx
df_raw.to_excel(new_data_path, index=False)

# Step 7: Report
print(f"✅ Done. {original_count - new_count} duplicates removed.")
print(f"📄 {new_count} unique rows remain in: {new_data_path}")



✅ Done. 2 duplicates removed.
📄 0 unique rows remain in: /content/drive/MyDrive/Colab Notebooks/CyberIncidents/new_data.xlsx


Data Extraction Using OpenAI API

In [38]:
# Import required libraries
import os
import re
import time
import json
import pandas as pd
from tqdm import tqdm
from docx import Document
from pdf2docx import Converter
from openai import OpenAI
from google.colab import userdata

In [39]:
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


In [40]:

client = OpenAI()  # Uses the env variable automatically

In [41]:
# Define schema for extraction
schema_description = """
You are a data extraction assistant who is an expert in cybersecurity incidents. For each text, extract the following fields:

1. **month** – The month the incident happened (month as text)
2. **year** - The year the incident happened
3. **source_country** - The country the attack originated from (if there are multiple source countries, separate them by comma; do not include continents or world regions)
4. **target_country** - The country the attack targeted (if there are multiple target countries, separate them by comma; do not include continents or world regions)
5. **target_region** - If source text indicates a specific continent or wold region (e.g., Southeast Asia) enter in this field
6. **threat_actor** - Name of the threat actor if known otherwise coded as unknown
7. **threat_actor_type** - such as state sponsored, cybercriminal, hacktivist, unknown,
8. **target_organization** - The organization or entity that was targeted
9. **target_sector** - industry type such as healthcare, finance, private, government, education, military, critical infrastructure, etc.
10. **attack_type** – Such as ransomware, DDoS, phishing, etc. indicating broad category of malicious activity
11. **attack_vector** - emails, phones, software vulnerabilities etc. specific path or method an attacker uses to exploit a system and deliver the attacks
12. **target_systems** such as telecommunication networks, linkedin profiles, web servers, email servers, government networks etc (try to standardize similar targets)
13. **impact** – What was affected (data stolen, systems taken offline, etc.)
14. **data_compromised** - true or false
15. **data_types**  - PII, intellectual property, government documents etc.
16. **financial_impact** - amount of impact in dollars - null if none indicated
17. **mitigation_steps** - can include more than one separated by comma
18. **information_source** - source that reports the incident

If no data is available in the text regarding the field of interest, then code as null

Respond in JSON for valid incidents. If text is about general recent trends or comparisons such as year over year:
{"type": "news"}
Return the JSON directly, without markdown formatting (no triple backticks)
"""

In [42]:
import json

def extract_structured_data(text, schema_description, parse_as_json=True):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": schema_description},
                {"role": "user", "content": text}
            ],
            temperature=0.2,
            top_p=1.0
        )

        # Extract the content
        result_text = response.choices[0].message.content.strip()

        if parse_as_json:
            try:
                return json.loads(result_text)
            except json.JSONDecodeError:
                print("⚠️ Output was not valid JSON. Returning raw text.")
                print("Model output:\n", result_text)
                return result_text
        else:
            return result_text

    except (AttributeError, IndexError) as e:
        print(f"❌ Unexpected response structure: {e}")
        print("Raw response:\n", response)
        return None

    except Exception as e:
        print(f"❌ General error during extraction: {e}")
        return None


In [43]:
# Load raw data from Excel
input_path = "/content/drive/MyDrive/Colab Notebooks/CyberIncidents/new_data.xlsx"
df_raw = pd.read_excel(input_path)

In [44]:
df_raw

,Event Description
0,April 2025: North Korean cyber spies are expan...


In [45]:
# check one more time df_raw includes only rows not processed before
try:
    # Attempt to load existing data
    df_existing = pd.read_excel(existing_path)

    # Extract unique existing raw texts from first column
    existing_texts = df_existing.iloc[:, 0].dropna().astype(str).unique().tolist()

    # Filter new data
    df_filtered = df_raw[~df_raw["Event Description"].isin(existing_texts)].reset_index(drop=True)
    print(f"🧹 Filtered: {len(df_filtered)} new records to process (from {len(df_raw)} total).")

except FileNotFoundError:
    print(f"⚠️ Warning: '{existing_path}' not found. Proceeding without duplicate filtering.")
    df_filtered = df_raw.copy()


🧹 Filtered: 0 new records to process (from 1 total).


In [46]:
from tqdm import tqdm
import json

structured_rows = []
raw_texts_structured = []
news_rows = []
non_parsed_texts = []

for i, text in enumerate(tqdm(df_filtered["Event Description"], desc="Processing")):
    print(f"\n🟡 Processing item {i+1}/{len(df_filtered)}")
    result = extract_structured_data(text, schema_description)

    if result:
        print(f"🔹 Raw result from GPT:\n{result}\n")
        try:
            # Parse JSON if needed
            data = result if isinstance(result, dict) else json.loads(result)

            if data.get("type") == "news":
                print("📰 Detected news entry.")
                news_rows.append(text)
            else:
                print("✅ Parsed structured data.")
                raw_texts_structured.append(text)
                structured_rows.append(data)

        except Exception as e:
            print(f"❌ JSON parsing failed: {e}")
            print(f"Result was:\n{result}")
            non_parsed_texts.append(text)
    else:
        print("⚠️ No result returned from model.")
        non_parsed_texts.append(text)


Processing: 0it [00:00, ?it/s]


In [47]:
# Convert to DataFrames

# Structured data - should always have rows if any were parsed
df_structured = pd.DataFrame(structured_rows)

# News data - safe creation even if empty
df_news = pd.DataFrame(news_rows, columns=["Event Description"]) if news_rows else pd.DataFrame(columns=["Event Description"])

# Non-parsed data - safe creation even if empty
df_non_parsed = pd.DataFrame(non_parsed_texts, columns=["Event Description"]) if non_parsed_texts else pd.DataFrame(columns=["Event Description"])

In [48]:
# save rows classified as news separately into news.xlsx file

news_path = "/content/drive/MyDrive/Colab Notebooks/CyberIncidents/news.xlsx"

if not df_news.empty:
    # If there is news data, append or create the file
    if os.path.exists(news_path):
        existing_news_df = pd.read_excel(news_path)
        combined_news = pd.concat([existing_news_df, df_news], ignore_index=True).drop_duplicates(subset=["Event Description"])
        combined_news.to_excel(news_path, index=False)
        print(f"✅ Appended {len(df_news)} news entries to {news_path}")
    else:
        df_news.to_excel(news_path, index=False)
        print(f"✅ Created news file: {news_path}")
else:
    print("⚠️ No news entries to save.")

⚠️ No news entries to save.


In [49]:
# save rows classified as non-parsed into non-parsed.xlsx

non_parsed_path = "/content/drive/MyDrive/Colab Notebooks/CyberIncidents/non-parsed.xlsx"

if not df_non_parsed.empty:
    if os.path.exists(non_parsed_path):
        existing_non_parsed_df = pd.read_excel(non_parsed_path)
        combined_non_parsed = pd.concat([existing_non_parsed_df, df_non_parsed], ignore_index=True).drop_duplicates(subset=["Event Description"])
        combined_non_parsed.to_excel(non_parsed_path, index=False)
        print(f"✅ Appended {len(df_non_parsed)} non-parsed entries to {non_parsed_path}")
    else:
        df_non_parsed.to_excel(non_parsed_path, index=False)
        print(f"✅ Created non-parsed file: {non_parsed_path}")
else:
    print("⚠️ No non-parsed entries to save.")

⚠️ No non-parsed entries to save.


In [50]:
# save parsed data into cyber-incidents-final.xlsx

def save_structured_data(
    extracted_df,
    raw_texts,
    output_path="/content/drive/MyDrive/Colab Notebooks/CyberIncidents/cyber-incidents-final.xlsx"
):
    # Ensure Event Description is first column with stripped strings
    extracted_df = extracted_df.copy()
    extracted_df.insert(0, "Event Description", pd.Series(raw_texts).astype(str).str.strip())

    # Save structured data to final file
    if not os.path.exists(output_path):
        extracted_df.to_excel(output_path, index=False)
        print(f"✅ Created new file: {output_path}")
    else:
        existing_df = pd.read_excel(output_path)
        existing_raw = existing_df["Event Description"].astype(str).str.strip()
        incoming_raw = extracted_df["Event Description"].astype(str).str.strip()

        is_new = ~incoming_raw.isin(existing_raw)
        new_unique_df = extracted_df[is_new]

        if not new_unique_df.empty:
            combined_df = pd.concat([existing_df, new_unique_df], ignore_index=True)
            combined_df.to_excel(output_path, index=False)
            print(f"✅ Appended {len(new_unique_df)} new rows to {output_path}")
        else:
            print("⚠️ No new unique structured entries to append.")


In [52]:
save_structured_data(df_structured, raw_texts_structured)

⚠️ No new unique structured entries to append.
